# Kelly Criterion Applied to SPY
### A Historical Risk-Return Case Study

## Overview

The amount of capital allocated and the level of leverage used directly affect the risk–return relationship. To examine this trade-off, this case study applies the Kelly criterion to determine how leverage could modify the historical risk and return of an investment in SPY.

The Kelly allocations are estimated using SPY's historical return and volatility together with a historical risk-free rate.

The analysis compares an unleveraged investment with Full, Half, and Quarter Kelly strategies using annualized return, volatility, the Sharpe ratio, and maximum drawdown.

### Note

This historical analysis does not represent a predictive model or an investment recommendation.

## 1. Data Preparation

This case study analyzes SPY, an exchange-traded fund (ETF) that tracks the S&P 500 Index. Historical daily price data from January 2016 to December 2025 is downloaded through the Yahoo Finance API.

Adjusted closing prices are used because they account for corporate actions such as dividend payments and stock splits, providing a more accurate representation of the investment's historical performance.

Daily simple returns and log returns are calculated from the adjusted closing prices. Simple returns are used to estimate SPY's expected return and volatility, apply the Kelly criterion, and evaluate the historical performance of each strategy. Log returns are retained to examine continuously compounded growth and compare it with the cumulative growth obtained from simple returns.

The risk-free rate is approximated using the 3-Month U.S. Treasury constant maturity rate obtained from the Federal Reserve Economic Data (FRED) over the same analysis period. Because the series is reported as an annualized yield expressed in percentage points, it is divided by 100 to convert it into decimal form.

The historical average of the annual risk-free-rate observations is used to estimate SPY's expected excess return and the Kelly allocations. For the historical strategy comparison, the risk-free-rate series is aligned with SPY's trading dates and converted into daily returns.

In [52]:
import yfinance as yf
import numpy as np
import plotly.express as px
from fredapi import Fred
from datetime import date
from config import FRED_API_KEY
import pandas as pd


# Define the analysis period
start_date = date(2016, 1, 1)
end_date = date(2026, 1, 1)
ticker = "SPY"


# Download historical SPY prices
data = yf.download(tickers=ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)
adj_prices = data["Close"].iloc[:, 0].rename("Adjusted Close")


# Calculate daily SPY returns
simple_returns = adj_prices.pct_change().dropna().rename("Simple Return")
log_returns = np.log(adj_prices / adj_prices.shift(1)).dropna().rename("Log Return")


# Download the historical risk-free rate from FRED
fred = Fred(api_key=FRED_API_KEY)
risk_free_series = fred.get_series("DGS3MO", observation_start=start_date, observation_end=end_date)

# Convert percentage points to decimal form
risk_free_series = risk_free_series.dropna().div(100).rename("Annual Risk-Free Rate")

# Calculate the historical average annual risk-free rate
risk_free_rate = risk_free_series.mean()

print(f"Average annual risk-free rate: {risk_free_rate:.2%}")


# Plot historical adjusted closing price
fig = px.line(
    x=adj_prices.index,
    y=adj_prices,
    title="<b>SPY Adjusted Closing Price (2016–2025)</b>",
    labels={
        "x": "Date",
        "y": "Adjusted Closing Price (USD)",
    },
)

fig.update_traces(
    line={
        "color": "navy",
        "width": 2,
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    title_x=0.5,
)

fig.show()

Average annual risk-free rate: 2.25%


### Insights

SPY followed a general upward trend from 2016 to 2025. The most substantial decline occurred during the COVID-19 market crash in early 2020, while other notable declines occurred in late 2018, throughout 2022, and in April 2025.

However, the price chart alone does not quantify the magnitude of these losses or allow investment strategies to be compared. Therefore, return and drawdown measures are needed.

## 2. Historical Return Analysis

Historical simple returns are used to estimate SPY's expected return and volatility. Logarithmic returns are used to examine continuously compounded growth and the effect of volatility on long-term compounding. The daily sample arithmetic mean return and sample volatility are calculated as follows:

$$\hat{\mu}_{daily}=\frac{1}{n}\sum_{t=1}^{n}R_t$$

$$\hat{\sigma}_{daily}=\sqrt{\frac{1}{n-1}\sum_{t=1}^{n}(R_t-\hat{\mu}_{daily})^2}$$

where:
- $R_t$ represents SPY's simple return on trading day $t$
- $n$ is the number of daily return observations
- $\hat{\mu}_{daily}$ is the estimated daily arithmetic mean return
- $\hat{\sigma}_{daily}$ is the estimated daily volatility.

In this analysis, volatility is used as a measure of the variability of SPY's returns and therefore as a proxy for investment risk.

### Daily Return Distribution

In [20]:
return_statistics = simple_returns.describe().to_frame(name="Daily Simple Return")
return_statistics

,Daily Simple Return
count,2513.000000
mean,0.000617
std,0.011344
min,-0.109424
25%,-0.003576
50%,0.000731
75%,0.005943
max,0.105019


In [62]:
# Calculate distribution statistics
return_mean = simple_returns.mean()
return_median = simple_returns.median()
return_skewness = simple_returns.skew()
return_kurtosis = simple_returns.kurt()  # Excess kurtosis

fig = px.histogram(
    simple_returns,
    x="Simple Return",
    nbins=80,
    title="<b>Distribution of SPY Daily Simple Returns</b>",
    labels={"Simple Return": "Daily Simple Return"},
    color_discrete_sequence=["#2563EB"],
)

fig.update_traces(
    marker_line_color="white",
    marker_line_width=0.5,
    opacity=0.85,
    hovertemplate=(
        "Daily Simple Return: %{x:.2%}<br>"
        "Number of Trading Days: %{y}"
        "<extra></extra>"
    ),
)

# Zero-return reference line
fig.add_vline(
    x=0,
    line_width=1.5,
    line_color="black",
)

# Mean
fig.add_vline(
    x=return_mean,
    line_width=2,
    line_dash="dash",
    line_color="#DC2626",
    annotation_text=f"Mean: {return_mean:.3%}",
    annotation_position="top right",
)

# Median
fig.add_vline(
    x=return_median,
    line_width=2,
    line_dash="dot",
    line_color="#16A34A",
    annotation_text=f"Median: {return_median:.3%}",
    annotation_position="top left",
)

# Statistical summary
fig.add_annotation(
    x=0.99,
    y=0.95,
    xref="paper",
    yref="paper",
    text=(
        "<b>Distribution Statistics</b><br>"
        f"Skewness: {return_skewness:.3f}<br>"
        f"Excess Kurtosis: {return_kurtosis:.3f}"
    ),
    showarrow=False,
    align="left",
    xanchor="right",
    yanchor="top",
    bgcolor="rgba(255, 255, 255, 0.9)",
    bordercolor="#D1D5DB",
    borderwidth=1,
    borderpad=8,
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    bargap=0.03,
    xaxis_title="Daily Simple Return",
    yaxis_title="Number of Trading Days",
)

fig.update_xaxes(
    tickformat=".1%",
    zeroline=False,
)

fig.show()

### Insights

SPY daily simple returns were concentrated near zero, with a positive mean of 0.062% and a median of 0.073%. The negative skewness (-0.321) indicates a somewhat heavier left tail, suggesting greater downside tail risk. The high excess kurtosis (14.733) reveals fat tails, meaning that extreme market movements occurred far more frequently than a normal distribution would predict.

### Annualized Estimates

To annualize the estimated arithmetic mean return and volatility, the following formulas are used:

$$\hat{\mu}_{annual}=252\hat{\mu}_{daily}$$

$$\hat{\sigma}_{annual}=\sqrt{252}\hat{\sigma}_{daily}$$

where $252$ represents the approximate number of trading days in a year. The annualized arithmetic mean return and volatility are used to estimate the continuous Kelly allocation.

In [46]:
daily_expected_return = simple_returns.mean()
daily_volatility = simple_returns.std()

trading_days = 252

annual_expected_return = daily_expected_return * trading_days
annual_volatility = daily_volatility * np.sqrt(trading_days)

print(f"Daily expected return: {daily_expected_return:.4%}")
print(f"Daily volatility: {daily_volatility:.4%}")

Daily expected return: 0.0617%
Daily volatility: 1.1344%


### Insights

SPY had an average daily return of 0.0617% and daily volatility of 1.1344%. This indicates that typical daily fluctuations were considerably larger than the average daily return.

On an annualized basis, SPY had an estimated arithmetic mean return of 15.54% and volatility of 18.01%. Therefore, SPY generated a positive historical average return during the analysis period, but its returns also showed substantial variability.

These values are historical estimates and do not imply that SPY earned 15.54% in every year or that it will produce the same return in the future.

### Simple and Logarithmic Return Comparison

Simple returns measure percentage changes in price, while logarithmic returns represent continuously compounded growth.

The arithmetic mean of simple returns is used to estimate the expected return required by the Kelly approximation. Logarithmic returns are used to examine compounded growth and demonstrate the relationship between the two return definitions.

In [48]:
annualized_mean_log_return = log_returns.mean() * trading_days

print(f"Annualized arithmetic mean return: {annual_expected_return:.2%}")
print(f"Annualized mean log return: {annualized_mean_log_return:.2%}")
print()

cumulative_return_simple = (1 + simple_returns).prod() - 1
cumulative_return_log = np.exp(log_returns.sum()) - 1

print(f"Cumulative return from simple returns: {cumulative_return_simple:.2%}")
print(f"Cumulative return from log returns: {cumulative_return_log:.2%}")

Annualized arithmetic mean return: 15.54%
Annualized mean log return: 13.91%

Cumulative return from simple returns: 300.25%
Cumulative return from log returns: 300.25%


### Insights

The annualized mean logarithmic return is lower than the annualized arithmetic mean return because volatility reduces compounded growth.

However, simple and logarithmic returns produce the same cumulative return when they are compounded correctly. Simple returns are compounded multiplicatively, whereas logarithmic returns are added and subsequently converted back using the exponential function:

$$\prod_{t=1}^{T}(1+R_t)-1=\exp\left(\sum_{t=1}^{T}r_t\right)-1$$

The arithmetic mean return continues to be used in the Kelly approximation, while logarithmic returns describe continuously compounded historical growth.

## 3. Kelly Criterion

### Kelly Allocation Estimation

The Kelly criterion determines the allocation to a risky asset that maximizes the expected long-term logarithmic growth rate of wealth. When a risk-free asset is available, SPY's expected return must be compared with the return that could be earned without assuming market risk.

Under the continuous-time approximation, the expected logarithmic growth rate of wealth is:

$$G(\ell)=r_f+\ell(\hat{\mu}-r_f)-\frac{1}{2}\ell^2\hat{\sigma}^2$$

where:

- $\ell$ is the fraction of wealth allocated to SPY.
- $\hat{\mu}$ is SPY's estimated annualized arithmetic mean return.
- $r_f$ is the estimated annual risk-free rate.
- $\hat{\sigma}$ is SPY's estimated annualized volatility.
- $1-\ell$ is the fraction allocated to or financed through the risk-free asset.

The value of $\ell$ determines how the investor's wealth is distributed:

- If $\ell<0$, the investor takes a short position in SPY.
- If $\ell=0$, all wealth is invested in the risk-free asset.
- If $0<\ell<1$, wealth is divided between SPY and the risk-free asset.
- If $\ell=1$, all wealth is invested in SPY without leverage.
- If $\ell>1$, the investor borrows at the risk-free rate to obtain leveraged exposure to SPY.

The expected logarithmic growth rate in excess of the risk-free investment can be written as:

$$R(\ell)=G(\ell)-r_f=\ell(\hat{\mu}-r_f)-\frac{1}{2}\ell^2\hat{\sigma}^2$$

Because $r_f$ is constant with respect to $\ell$, maximizing $R(\ell)$ produces the same optimal allocation as maximizing the total logarithmic growth function $G(\ell)$.

### Deriving the Optimal Allocation

To find the allocation that maximizes the expected logarithmic growth rate, the excess-growth function is differentiated with respect to $\ell$:

$$R'(\ell)=(\hat{\mu}-r_f)-\ell\hat{\sigma}^2$$

The first-order condition is obtained by setting the derivative equal to zero:

$$(\hat{\mu}-r_f)-\ell\hat{\sigma}^2=0$$

Solving for $\ell$ gives the estimated Full Kelly allocation:

$$\hat{\ell}^*=\frac{\hat{\mu}-r_f}{\hat{\sigma}^2}$$

The second derivative is:

$$R''(\ell)=-\hat{\sigma}^2<0$$

Because the second derivative is negative, the growth function is strictly concave. Therefore, $\hat{\ell}^*$ represents the unique allocation that maximizes the expected logarithmic growth rate under the assumptions of the model.

The formula without a risk-free rate,

$$\hat{\ell}^*=\frac{\hat{\mu}}{\hat{\sigma}^2}$$

is a special case of the general Kelly formula in which $r_f=0$.

In [49]:
def expected_excess_growth(leverage, expected_return, volatility, risk_free_rate):
    """Estimate the expected excess logarithmic growth rate."""
    return leverage * (expected_return - risk_free_rate) - 0.5 * leverage**2 * volatility**2

def kelly_leverage(expected_return, volatility, risk_free_rate):
    """
    Calculate the Full Kelly allocation.

    ℓ* = (μ - rf) / σ²
    """
    return (expected_return - risk_free_rate) / volatility**2

full_kelly = kelly_leverage(annual_expected_return, annual_volatility, risk_free_rate)

half_kelly = 0.50 * full_kelly
quarter_kelly = 0.25 * full_kelly

kelly_leverages = {
    "Quarter Kelly": quarter_kelly,
    "Half Kelly": half_kelly,
    "Full Kelly": full_kelly,
}

kelly_growth_rates = {}

for strategy, leverage in kelly_leverages.items():
    kelly_growth_rates[strategy] = expected_excess_growth(leverage, annual_expected_return, annual_volatility, risk_free_rate)

print(f"Annual expected return: {annual_expected_return:.2%}")
print(f"Annual volatility: {annual_volatility:.2%}")
print(f"Annual risk-free rate: {risk_free_rate:.2%}")
print()

for strategy, leverage in kelly_leverages.items():
    print(
        f"{strategy}: "
        f"{leverage:.2f}x SPY allocation | "
        f"{kelly_growth_rates[strategy]:.2%} "
        "expected excess growth"
    )

Annual expected return: 15.54%
Annual volatility: 18.01%
Annual risk-free rate: 2.25%

Quarter Kelly: 1.02x SPY allocation | 11.91% expected excess growth
Half Kelly: 2.05x SPY allocation | 20.42% expected excess growth
Full Kelly: 4.10x SPY allocation | 27.23% expected excess growth


In [50]:
# Excess growth returns to zero at twice the Full Kelly allocation.
break_even_leverage = 2 * full_kelly

# SPY allocations from zero to slightly beyond the break-even point.
leverage_range = np.linspace(
    0,
    break_even_leverage * 1.05,
    300,
)

growth_rates = expected_excess_growth(
    leverage_range,
    annual_expected_return,
    annual_volatility,
    risk_free_rate,
)

fig = px.line(
    x=leverage_range,
    y=growth_rates,
    title="<b>Expected Excess Growth at Different SPY Allocations</b>",
    labels={
        "x": "SPY Allocation",
        "y": "Expected Excess Logarithmic Growth Rate",
    },
)

fig.update_traces(
    hovertemplate=(
        "SPY Allocation: %{x:.2f}x<br>"
        "Expected Excess Growth: %{y:.2%}"
        "<extra></extra>"
    )
)

fig.add_vline(
    x=1,
    line_dash="dot",
    line_color="gray",
    annotation_text="Unleveraged: 1.00x",
    annotation_position="bottom left",
)

kelly_colors = {
    "Quarter Kelly": "seagreen",
    "Half Kelly": "darkorange",
    "Full Kelly": "navy",
}

annotation_positions = {
    "Quarter Kelly": "top left",
    "Half Kelly": "top left",
    "Full Kelly": "top right",
}

for strategy, leverage in kelly_leverages.items():
    growth_rate = kelly_growth_rates[strategy]

    fig.add_vline(
        x=leverage,
        line_dash="dash",
        line_color=kelly_colors[strategy],
        annotation_text=f"{strategy}: {leverage:.2f}x",
        annotation_position=annotation_positions[strategy],
    )

    fig.add_scatter(
        x=[leverage],
        y=[growth_rate],
        mode="markers",
        marker={
            "color": kelly_colors[strategy],
            "size": 10,
        },
        hovertemplate=(
            f"<b>{strategy}</b><br>"
            "SPY Allocation: %{x:.2f}x<br>"
            "Expected Excess Growth: %{y:.2%}"
            "<extra></extra>"
        ),
        name=strategy,
    )

fig.add_vline(
    x=break_even_leverage,
    line_dash="dash",
    line_color="firebrick",
    annotation_text=f"Break-even: {break_even_leverage:.2f}x",
    annotation_position="bottom right",
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    hovermode="x unified",
    showlegend=False,
    yaxis_tickformat=".1%",
)

fig.show()

### Insights

The estimated Full Kelly allocation is 4.10x, implying 410% exposure to SPY and 310% financed through borrowing. At this allocation, expected excess logarithmic growth reaches its theoretical maximum of 27.23%.

Fractional Kelly strategies reduce leverage while retaining much of the potential growth. Half Kelly, for example, uses half the exposure but preserves 75% of the maximum expected excess growth. Growth falls beyond Full Kelly and reaches zero at the 8.20x break-even allocation.

These estimates are highly sensitive to expected return and volatility and rely on unrealistic assumptions, including stable parameters and borrowing at the risk-free rate without costs or constraints. Therefore, Full Kelly should be viewed as a theoretical benchmark rather than an investment recommendation.

## 4. Historical Strategy Comparison

To examine the historical consequences of the estimated allocations, the unleveraged, Quarter Kelly, Half Kelly, and Full Kelly strategies are compared using fixed daily-rebalanced allocations.

For each strategy, the fraction allocated to SPY earns SPY's daily simple return. The remaining fraction is invested in or financed through the historical risk-free rate:

$$R_{p,t}=\ell R_{\text{SPY},t}+(1-\ell)R_{f,t}$$

where $\ell$ represents the allocation to SPY.

Because the Kelly allocations are estimated using the entire 2016–2025 sample, this is an in-sample historical comparison rather than an implementable out-of-sample backtest.

In [59]:
# Align the annual risk-free-rate series with SPY trading dates
combined_index = risk_free_series.index.union(simple_returns.index)

aligned_annual_risk_free_rate = (
    risk_free_series
    .reindex(combined_index)
    .sort_index()
    .ffill()
    .reindex(simple_returns.index)
    .bfill()
)

# Convert the annual risk-free yield into an effective daily return
daily_risk_free_return = ((1 + aligned_annual_risk_free_rate) ** (1 / trading_days)- 1)

# Fixed SPY allocations used in the historical comparison
strategy_allocations = {
    "Unleveraged": 1.0,
    "Quarter Kelly": quarter_kelly,
    "Half Kelly": half_kelly,
    "Full Kelly": full_kelly,
}

# Calculate daily returns for each strategy
strategy_returns = pd.DataFrame(index=simple_returns.index)

for strategy, allocation in strategy_allocations.items():
    strategy_returns[strategy] = (allocation * simple_returns+ (1 - allocation) * daily_risk_free_return)

strategy_returns = strategy_returns.reset_index()
strategy_returns.head()

,Date,Unleveraged,Quarter Kelly,Half Kelly,Full Kelly
0,2016-01-05,0.001691,0.001733,0.003457,0.006907
1,2016-01-06,-0.012615,-0.012925,-0.025858,-0.051724
2,2016-01-07,-0.023991,-0.024580,-0.049168,-0.098344
3,2016-01-08,-0.010977,-0.011247,-0.022502,-0.045012
4,2016-01-11,0.000990,0.001014,0.002021,0.004033


In [57]:
def calculate_strategy_metrics(returns, daily_risk_free_return):
    """Calculate historical performance and risk metrics."""

    observations = len(returns)

    cumulative_growth = (1 + returns).prod()

    cagr = cumulative_growth ** (trading_days / observations) - 1

    annualized_volatility = (returns.std() * np.sqrt(trading_days))

    annualized_excess_return = ((returns - daily_risk_free_return).mean()* trading_days)

    sharpe_ratio = (annualized_excess_return / annualized_volatility)

    wealth_index = (1 + returns).cumprod()
    drawdown = wealth_index / wealth_index.cummax() - 1
    maximum_drawdown = drawdown.min()

    return {
        "CAGR": cagr,
        "Annualized Volatility": annualized_volatility,
        "Sharpe Ratio": sharpe_ratio,
        "Maximum Drawdown": maximum_drawdown
    }


strategy_metrics = {
    strategy: calculate_strategy_metrics(strategy_returns[strategy], daily_risk_free_return)
    for strategy in strategy_returns.columns
}

strategy_summary = pd.DataFrame(strategy_metrics).T

formatted_summary = strategy_summary.copy()

percentage_columns = [
    "CAGR",
    "Annualized Volatility",
    "Maximum Drawdown"
]

for column in percentage_columns:
    formatted_summary[column] = formatted_summary[column].map(lambda value: f"{value:.2%}")

formatted_summary["Sharpe Ratio"] = formatted_summary["Sharpe Ratio"].map(lambda value: f"{value:.2f}")

formatted_summary

,CAGR,Annualized Volatility,Sharpe Ratio,Maximum Drawdown
Unleveraged,14.92%,18.01%,0.74,-33.72%
Quarter Kelly,15.20%,18.45%,0.74,-34.43%
Half Kelly,25.42%,36.90%,0.74,-59.81%
Full Kelly,33.43%,73.80%,0.74,-88.16%


In [ ]:
# Calculate the historical growth of one dollar
cumulative_wealth = (1 + strategy_returns).cumprod()

initial_wealth = pd.DataFrame(
    [np.ones(len(cumulative_wealth.columns))],
    index=[adj_prices.index[0]],
    columns=cumulative_wealth.columns
)

cumulative_wealth = pd.concat([initial_wealth, cumulative_wealth])

cumulative_wealth_long = (
    cumulative_wealth
    .rename_axis("Date")
    .reset_index()
    .melt(id_vars="Date", var_name="Strategy", value_name="Growth of $1")
)

fig = px.line(
    cumulative_wealth_long,
    x="Date",
    y="Growth of $1",
    color="Strategy",
    title="<b>Historical Growth of $1 by Kelly Strategy</b>"
)

fig.update_traces(
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Date: %{x|%Y-%m-%d}<br>"
        "Value: $%{y:.2f}"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    hovermode="x unified",
    yaxis_tickprefix="$"
)

fig.show()

### Insights

Higher SPY exposure increased both returns and risk. The unleveraged strategy returned 14.92% annually with an 18.01% volatility and a −33.72% maximum drawdown, while Full Kelly achieved the highest return at 33.43% but suffered 73.80% volatility and an extreme −88.16% drawdown.

All strategies had a similar Sharpe ratio of approximately 0.74 because leverage proportionally increased both excess return and volatility. Quarter Kelly was nearly identical to the unleveraged strategy, while Half Kelly offered higher growth but a substantially larger −59.81% drawdown.

Although Full Kelly maximized historical growth, its drawdowns make it impractical. These in-sample results also exclude realistic borrowing costs, transaction costs, taxes, and margin constraints, so they should be viewed as an illustration rather than an implementable strategy or investment recommendation.